# Aprovisionamiento de Hardware, Conexión a Drive e Ingesta de Datos

Este bloque verifica la asignación del acelerador gráfico y monta el sistema de archivos persistente. Es obligatorio ejecutar este cuaderno en un entorno con GPU. Una vez montado, se ingesta el corpus maestro y se extrae el vector de etiquetas operativas puras para el orquestador neuronal.

In [1]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
from google.colab import drive

# Auditoría de Hardware
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if dispositivo.type == 'cpu':
    raise SystemError("CRÍTICO: Entorno sin GPU detectado. Deteniendo ejecución para evitar saturación de memoria RAM.")
else:
    print(f"Acelerador gráfico activo: {torch.cuda.get_device_name(0)}")

# Conexión persistente a Google Drive
print("\nConectando con Google Drive...")
drive.mount('/content/drive')

# Rutas persistentes
BASE_DIR = Path('/content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR')
RUTA_GOLD = BASE_DIR / "data" / "gold" / "corpus_sitor_limpio.parquet"
DIR_MODELOS = BASE_DIR / "src" / "models" / "roberta_corporativo"
DIR_METRICAS = BASE_DIR / "metrics"

DIR_MODELOS.mkdir(parents=True, exist_ok=True)
DIR_METRICAS.mkdir(parents=True, exist_ok=True)

if not RUTA_GOLD.exists():
    raise FileNotFoundError(f"No se encuentra la ruta: {RUTA_GOLD}.")

# Ingesta
df = pd.read_parquet(RUTA_GOLD)
print(f"\nCorpus inyectado directamente desde Drive.")
print(f"Volumen: {len(df)} registros")

# Codificación de la Tripleta Objetivo
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['target_tripleta'])
print(f"Clases únicas a predecir: {len(le.classes_)}")

Acelerador gráfico activo: NVIDIA A100-SXM4-80GB

Conectando con Google Drive...
Mounted at /content/drive

Corpus inyectado directamente desde Drive.
Volumen: 15403 registros
Clases únicas a predecir: 56


# Tokenización y Construcción del Dataset (PyTorch)

Los modelos Transformer no consumen matrices dispersas (TF-IDF), sino secuencias densas de tokens enteros. Instanciamos el tokenizador de `roberta-base`.

Se define una clase heredada de `torch.utils.data.Dataset` para la ingesta eficiente. Fijamos la longitud de contexto en `MAX_LEN = 256` truncando el texto sobrante. En el dominio de soporte técnico, la anomalía o urgencia suele describirse en las primeras líneas; 256 tokens capturan esta señal sin desperdiciar VRAM. Además, exportamos el mapeo de clases para garantizar la trazabilidad operativa en inferencia.

In [2]:
import json
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset

# 1. Extracción y sellado del mapeo de clases (Imprescindible para producción)
id2label = {int(idx): str(label) for idx, label in enumerate(le.classes_)}
label2id = {v: k for k, v in id2label.items()}

with open(DIR_MODELOS / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(id2label, f, indent=4)
print(f"Diccionario de etiquetas (56 clases) sellado en: {DIR_MODELOS / 'label_mapping.json'}")

# 2. Descarga del Tokenizador
print("\nDescargando tokenizador: roberta-base...")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
# Guardamos también el tokenizador en nuestro directorio para inferencia offline
tokenizer.save_pretrained(DIR_MODELOS)
print("Tokenizador guardado localmente.")

# 3. Construcción del motor de ingesta (Dataset)
class SitorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts.tolist() if isinstance(texts, pd.Series) else texts
        self.labels = labels.tolist() if isinstance(labels, pd.Series) else labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        texto = str(self.texts[idx])
        etiqueta = self.labels[idx]

        # Tokenización on-the-fly
        encoding = self.tokenizer(
            texto,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(etiqueta, dtype=torch.long)
        }

print("\nClase SitorDataset definida correctamente.")

Diccionario de etiquetas (56 clases) sellado en: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/src/models/roberta_corporativo/label_mapping.json

Descargando tokenizador: roberta-base...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizador guardado localmente.

Clase SitorDataset definida correctamente.


# Orquestador Neuronal K-Fold con Validación Anidada (Fuga Cero y OOM Safe)

Para mantener la dinámica de convergencia óptima sin incurrir en fugas de datos (Data Leakage), se implementa una arquitectura de validación anidada.

La partición principal del `StratifiedGroupKFold` aísla un conjunto *Out-Of-Fold* (OOF) estrictamente ciego. Internamente, la porción de entrenamiento se sub-divide (90/10). El 10% resultante se inyecta como conjunto de monitoreo para el `EarlyStoppingCallback`. Se ha añadido una lógica de repliegue (fallback) en esta subdivisión: si el desbalanceo extremo deja a la clase minoritaria con menos de dos instancias en el pliegue, se desactiva temporalmente la estratificación para evitar colapsos (ValueError) en Scikit-Learn.

A nivel de negocio, el orquestador monitorea el `eval_accuracy` en lugar de la pérdida cruzada, priorizando la automatización del volumen masivo de tickets.

A nivel de infraestructura, el bucle incluye un limpiador explícito de memoria de vídeo (vaciado de caché CUDA y recolección de basura de PyTorch) al final de cada iteración, garantizando que los tensores residuales no provoquen un error `Out of Memory` (OOM) en la A100.

Finalmente, las métricas de rendimiento real se extraen evaluando el modelo restaurado contra el conjunto OOF virgen, reportando estimadores insesgados (cuasi-desviación estándar) para auditar la estabilidad operativa tanto en clases mayoritarias como en el long-tail.

In [3]:
import gc
import torch
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(cv.split(df['full_text'], df['label_id'], groups=df['cluster_id']))

metricas_cv = []
EPOCHS_FINAL = 20

for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n{'='*50}")
    print(f" Iniciando Entrenamiento Anidado - Fold {fold + 1}/5")
    print(f"{'='*50}")

    train_data_full = df.iloc[train_idx]
    val_data_oof = df.iloc[val_idx]

    # Lógica de fallback para proteger contra el desbalanceo extremo
    min_class_count = train_data_full['label_id'].value_counts().min()

    if min_class_count < 2:
        print(f" [!] Fallback activo: La clase más rara tiene {min_class_count} instancias.")
        print(f" [!] Desactivando estratificación en el particionado interno.")
        train_inner, val_inner = train_test_split(
            train_data_full,
            test_size=0.10,
            random_state=42
        )
    else:
        train_inner, val_inner = train_test_split(
            train_data_full,
            test_size=0.10,
            stratify=train_data_full['label_id'],
            random_state=42
        )

    print(f"Volumetrías del Fold:")
    print(f" - Entrenamiento puro: {len(train_inner)} tickets")
    print(f" - Monitoreo interno (Early Stopping): {len(val_inner)} tickets")
    print(f" - Evaluación ciega (OOF): {len(val_data_oof)} tickets")

    train_dataset = SitorDataset(train_inner['full_text'], train_inner['label_id'], tokenizer)
    val_inner_dataset = SitorDataset(val_inner['full_text'], val_inner['label_id'], tokenizer)
    val_oof_dataset = SitorDataset(val_data_oof['full_text'], val_data_oof['label_id'], tokenizer)

    modelo = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=len(le.classes_)
    ).to(dispositivo)

    # Configuración de entrenamiento orientada a volumen (Accuracy)
    args = TrainingArguments(
        output_dir=str(DIR_MODELOS / f"checkpoints_fold_{fold + 1}"),
        num_train_epochs=EPOCHS_FINAL,
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy", # CORRECCIÓN: Optimizar a volumen
        logging_steps=100,
        save_total_limit=1,
        report_to="none"
    )

    trainer = Trainer(
        model=modelo,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_inner_dataset,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        compute_metrics=compute_metrics
    )

    print("\nOptimizando pesos...")
    trainer.train()

    print("\nEvaluando partición OOF (Verdadera capacidad de generalización)...")
    metricas = trainer.evaluate(eval_dataset=val_oof_dataset)
    print(metricas)
    metricas_cv.append(metricas)

    if fold == 4:
        ruta_final = str(DIR_MODELOS / "roberta_corporativo_final")
        trainer.save_model(ruta_final)
        print(f"\nModelo de producción sellado en: {ruta_final}")

    # =====================================================================
    # PREVENCIÓN DE COLAPSO (OOM) EN PYTORCH
    # =====================================================================
    del modelo
    del trainer
    del train_dataset
    del val_inner_dataset
    del val_oof_dataset
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# AUDITORÍA FORENSE FINAL
# =====================================================================

acc_folds = [m['eval_accuracy'] for m in metricas_cv]
f1_macro_folds = [m['eval_f1_macro'] for m in metricas_cv]

acc_mean = np.mean(acc_folds)
acc_std = np.std(acc_folds, ddof=1)

f1_mean = np.mean(f1_macro_folds)
f1_std = np.std(f1_macro_folds, ddof=1)

print("\n" + "="*50)
print(" RENDIMIENTO ESTADÍSTICO (K-FOLD CV)")
print("="*50)
print(f"Accuracy Medio K-Fold:  {acc_mean:.4f} ± {acc_std:.4f}")
print(f"F1-Macro Medio K-Fold:  {f1_mean:.4f} ± {f1_std:.4f}")

inestabilidad_detectada = False

if acc_std > 0.05:
    print("\nADVERTENCIA (Mayoritaria): Alta varianza en Accuracy. Inestabilidad en la predicción de volumen.")
    inestabilidad_detectada = True

if f1_std > 0.05:
    print("\nADVERTENCIA (Minoritaria): Alta varianza en F1-Macro. La red es errática aprendiendo las clases raras.")
    inestabilidad_detectada = True

if not inestabilidad_detectada:
    print("\nESTABILIDAD CONFIRMADA: La red neuronal converge de forma robusta tanto en clases mayoritarias como en el long-tail.")


 Iniciando Entrenamiento Anidado - Fold 1/5
Volumetrías del Fold:
 - Entrenamiento puro: 11092 tickets
 - Monitoreo interno (Early Stopping): 1233 tickets
 - Evaluación ciega (OOF): 3078 tickets


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...


Evaluando partición OOF (Verdadera capacidad de generalización)...


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.116623,2.066460,20,0.602655,0.530454,0.600438


{'eval_loss': 2.066459894180298, 'eval_accuracy': 0.6026554404145078, 'eval_f1_macro': 0.5304542144658183, 'eval_f1_weighted': 0.6004383869271307}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modelo de producción sellado en: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/src/models/roberta_corporativo/roberta_corporativo_final

 RENDIMIENTO ESTADÍSTICO (K-FOLD CV)
Accuracy Medio K-Fold:  0.5882 ± 0.0099
F1-Macro Medio K-Fold:  0.5459 ± 0.0182

ESTABILIDAD CONFIRMADA: La red neuronal converge de forma robusta tanto en clases mayoritarias como en el long-tail.
